In [2]:
import os
import arcpy

In [3]:
out_gdb = r'C:\Users\colet\Documents\GEOG5890_Capstone\GEOG5890_Capstone_GDB\GeoAI_Capstone.gdb'

roads = r'C:\Users\colet\Documents\GEOG5890_Capstone\data\Denver_Roads_CDOT\Denver_Roads_Combined.shp'
crash_grid = os.path.join(out_gdb, 'Crash_Grid')

roads_proj = os.path.join(out_gdb, 'Roads_Proj')
arcpy.management.Project(roads, roads_proj, arcpy.Describe(crash_grid).spatialReference)

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Roads_Proj'>

In [4]:
road_grid = os.path.join(out_gdb, 'Road_Grid_Intersect')

arcpy.analysis.Intersect([crash_grid, roads_proj], road_grid, join_attributes = 'ALL', output_type = 'LINE')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Road_Grid_Intersect'>

In [5]:
arcpy.management.AddField(road_grid, 'RoadLength_m', 'DOUBLE')
arcpy.management.CalculateGeometryAttributes(road_grid, [['RoadLength_m', 'LENGTH']], length_unit = 'METERS')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Road_Grid_Intersect'>

In [7]:
road_length_table = os.path.join(out_gdb, 'Road_Length_by_Grid')
arcpy.analysis.Statistics(road_grid, road_length_table, [['RoadLength_m', 'SUM']], 'Grid_ID')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Road_Length_by_Grid'>

In [10]:
arcpy.management.AddField(crash_grid, 'Area_m2', 'DOUBLE')
arcpy.management.CalculateGeometryAttributes(crash_grid, [['Area_m2', 'AREA']], area_unit = 'SQUARE_METERS')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid'>

In [11]:
arcpy.management.JoinField(crash_grid, 'Grid_ID', road_length_table, 'Grid_ID', ['SUM_RoadLength_m'])

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid'>

In [17]:
arcpy.management.CalculateField(crash_grid, 'SUM_RoadLength_m',
                                '0 if !SUM_RoadLength_m! == None else !SUM_RoadLength_m!', 'PYTHON3')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid'>

In [18]:
arcpy.management.AddField(crash_grid, 'Road_Density', 'DOUBLE')
arcpy.management.CalculateField(crash_grid, 'Road_Density', '(!SUM_RoadLength_m! / !Area_m2!) * 1000', 'PYTHON3')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid'>